# 04 · Fan-Log Duty–RPM Calibration

**Purpose.** Parse the raw multimeter/tachometer log (`fan_log.csv`, UTF-16 encoded with junk header lines), detect the up-ramp/down-ramp segments, filter out noisy readings, and fit a linear duty–RPM relationship.

This is the final, cumulative version of a four-iteration development chain (parse → detect ramps → filter & style → add linear fit). The three earlier iterations are preserved in the Experiments notebook since each added a capability that is fully included in this final cell.

## Parse, filter, ramp-split, and fit fan log data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Read CSV ---
with open("/content/fan_log.csv", "r", encoding="utf-16") as f:
    lines = f.readlines()

# --- Extract numeric duty and rpm ---
data = []
for line in lines:
    line = line.strip()
    if (
        line == "" or
        line.startswith("---") or
        "Available" in line or
        "Duty" in line
    ):
        continue

    cleaned = ''.join(c if c.isdigit() or c in [',', '.'] else '' for c in line)
    if ',' in cleaned:
        try:
            duty, rpm = map(float, cleaned.split(',', 1))
            data.append([duty, rpm])
        except ValueError:
            pass

df = pd.DataFrame(data, columns=["duty", "rpm"])

# --- Keep multiples of 10 ---
df = df[df['duty'] % 10 == 0]

# --- RPM cutoff ---
df = df[df['rpm'] <= 1400]

# --- Ramp detection ---
df['ramp'] = 'up'
peak_idx = df['duty'].idxmax()
df.loc[peak_idx + 1:, 'ramp'] = 'down'

# --- Average ---
avg_df = (
    df.groupby(['ramp', 'duty'], as_index=False)
      .mean()
      .sort_values('duty')
)

# --- Plot ---
plt.figure(figsize=(9,6))

for ramp, style in zip(['up', 'down'], ['-', '--']):
    subset = avg_df[avg_df['ramp'] == ramp]

    x = subset['duty'].values
    y = subset['rpm'].values

    # ----- LEAST SQUARES LINEAR FIT -----
    b, a = np.polyfit(x, y, 1)   # least squares

    x_fit = np.linspace(x.min(), x.max(), 300)
    y_fit = a + b * x_fit

    # Data
    plt.plot(x, y, 'o', markersize=8, label=f"{ramp.capitalize()} Ramp Data")

    # Fit
    plt.plot(
        x_fit, y_fit,
        style,
        linewidth=3,
        label=f"{ramp.capitalize()} Ramp Fit"
    )

# --- Formatting ---
plt.xticks(range(0, 101, 10), fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')

plt.xlabel("Duty Cycle (%)", fontsize=18, fontweight='bold')
plt.ylabel("RPM", fontsize=18, fontweight='bold')
plt.title("Duty Cycle vs RPM",
          fontsize=20, fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=13)
plt.tight_layout()
plt.show()